In [1]:
# Diagnostic cell — run this in one cell to confirm envs and test AzureChatOpenAI
from dotenv import load_dotenv
import os, traceback
from langchain_openai import AzureChatOpenAI 
from typing import TypedDict
from langgraph.graph import StateGraph,START,END

load_dotenv()  # ensure .env loaded

print("AZURE_OPENAI_KEY present:", bool(os.getenv("AZURE_OPENAI_KEY")))
print("AZURE_OPENAI_ENDPOINT:", os.getenv("AZURE_OPENAI_ENDPOINT"))
print("AZURE_OPENAI_DEPLOYMENT:", os.getenv("AZURE_OPENAI_DEPLOYMENT"))
print("AZURE_OPENAI_API_VERSION:", os.getenv("AZURE_OPENAI_API_VERSION"))
print("OPENAI_API_KEY (if set):", bool(os.getenv("OPENAI_API_KEY")))

# Map env names the SDK might look for (non-destructive)
if os.getenv("AZURE_OPENAI_KEY") and not os.getenv("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_KEY")
if os.getenv("AZURE_OPENAI_KEY") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_KEY")

# Instantiate explicitly with api_key to avoid env name issues
try:
    model = AzureChatOpenAI(
        azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
        temperature=0.2,
        api_key=os.getenv("AZURE_OPENAI_KEY"),  # explicit
    )
    print('AzureChatOpenAI instantiated OK')
except Exception as e:
    print('Failed to instantiate AzureChatOpenAI:')
    traceback.print_exc()

# Try a short call (small tokens) to test end-to-end
try:
    resp = model.invoke("Say hello in one short sentence.")
    print("Invoke result:", getattr(resp, "content", None) or resp)
except Exception as e:
    print('Error during invoke:')
    traceback.print_exc()

AZURE_OPENAI_KEY present: True
AZURE_OPENAI_ENDPOINT: https://crr-openai-esg.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT: gpt-4o-mini
AZURE_OPENAI_API_VERSION: 2024-12-01-preview
OPENAI_API_KEY (if set): False
AzureChatOpenAI instantiated OK
Invoke result: Hello!


In [41]:
class BlogState(TypedDict):
    title   : str
    outline : str 
    content : str
    

In [ ]:
def create_outline(state:BlogState)->BlogState:
    #fetch title
    title = state['title']

    #call llm gen outline
    prompt = f'Generate a detatiled outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    #update state
    state['outline'] = outline

    return state



In [43]:
def create_blog(state:BlogState)->BlogState:
    title = state['title']
    outline = state['outline']
    promt = f'write a detail blog blog on the title -{title} usign the following outline \n {outline}'

    content = model.invoke(promt).content
    state['content']= content
    return state

In [44]:
graph = StateGraph(BlogState)

#node
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

#add edeges

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

#compile the graph
workflow= graph.compile()



In [45]:
initial_state = {'title':'How cook butter chicken?'}
finalstate = workflow.invoke(initial_state)
print(finalstate['content'])
print("\n\n")
print(finalstate['outline'])



# How to Cook Butter Chicken: A Delicious Journey into Indian Cuisine

## Introduction

Butter Chicken, or Murgh Makhani, is a beloved dish that has captured the hearts and taste buds of food lovers around the globe. Originating from Delhi, this creamy, rich curry is a staple in Indian cuisine and has become a favorite in restaurants and homes alike. Its unique blend of spices and flavors makes it a standout dish that is both comforting and indulgent.

In this blog, we will explore the art of cooking Butter Chicken, covering everything from the essential ingredients and preparation methods to cooking techniques and serving suggestions. Whether you’re a seasoned chef or a beginner in the kitchen, this guide will help you create a mouthwatering Butter Chicken that will impress your family and friends.

## Section 1: Understanding Butter Chicken

### Explanation of Butter Chicken (Murgh Makhani)

Butter Chicken is characterized by its creamy texture, rich flavor, and a perfect balance of 